# Notebook 5: Advanced Features - Taking Full Advantage

## Welcome to Advanced Trading! 🚀

You've mastered DQN, PPO, and SAC. Now let's unlock the **full power** of the TradeMaster platform with advanced features!

### What you'll learn:
1. **Custom technical indicators** for better signals
2. **Risk management strategies** (position sizing, stop-loss)
3. **Reward function engineering** for better learning
4. **Hyperparameter tuning** techniques
5. **Multi-contract trading** strategies
6. **Performance analysis** tools
7. **Production deployment** tips

This is where beginners become experts! 💪

## Part 1: Advanced Technical Indicators

### Beyond Basic Indicators

The default indicators are good, but let's add more powerful ones!

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

print("✅ Libraries imported!")

### Custom Indicator Functions

In [ ]:
def add_advanced_indicators(df):
    """
    Add advanced technical indicators to enhance RL agent's decision making
    """
    df = df.copy()
    
    # 1. RSI (Relative Strength Index)
    def calculate_rsi(series, period=14):
        delta = series.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / (loss + 1e-8)
        return 100 - (100 / (1 + rs))
    
    df['rsi'] = calculate_rsi(df['close'])
    df['rsi_normalized'] = (df['rsi'] - 50) / 50  # Normalize to [-1, 1]
    
    # 2. MACD (Moving Average Convergence Divergence)
    exp1 = df['close'].ewm(span=12, adjust=False).mean()
    exp2 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd'] = exp1 - exp2
    df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_hist'] = df['macd'] - df['macd_signal']
    
    # Normalize MACD
    df['macd_normalized'] = (df['macd'] - df['macd'].rolling(100).mean()) / (df['macd'].rolling(100).std() + 1e-8)
    
    # 3. Bollinger Bands
    df['bb_middle'] = df['close'].rolling(window=20).mean()
    bb_std = df['close'].rolling(window=20).std()
    df['bb_upper'] = df['bb_middle'] + (bb_std * 2)
    df['bb_lower'] = df['bb_middle'] - (bb_std * 2)
    df['bb_position'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'] + 1e-8)
    
    # 4. ATR (Average True Range) - Volatility
    high_low = df['high'] - df['low']
    high_close = abs(df['high'] - df['close'].shift())
    low_close = abs(df['low'] - df['close'].shift())
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df['atr'] = true_range.rolling(window=14).mean()
    df['atr_normalized'] = df['atr'] / df['close']  # As % of price
    
    # 5. Volume indicators
    df['volume_ratio'] = df['volume'] / df['volume'].rolling(window=20).mean()
    df['volume_trend'] = df['volume'].pct_change(5)
    
    # 6. Price momentum (multiple timeframes)
    for period in [3, 7, 14, 21, 30]:
        df[f'momentum_{period}'] = df['close'].pct_change(period)
    
    # Fill NaN values
    df = df.bfill().fillna(0)
    
    return df

# Example usage
CONTRACT = 'MCL-1m'
DATA_DIR = Path('./data') / CONTRACT

# Load sample data
sample_df = pd.read_csv(DATA_DIR / 'train.csv', index_col=0).head(5000)
enhanced_df = add_advanced_indicators(sample_df)

print("✅ Advanced indicators added!")
print(f"\n📊 New indicators:")
new_cols = [col for col in enhanced_df.columns if col not in sample_df.columns]
for col in new_cols:
    print(f"  - {col}")

### Visualize Advanced Indicators

In [ ]:
# Plot a sample period
plot_df = enhanced_df.iloc[1000:1500].reset_index(drop=True)

fig, axes = plt.subplots(4, 1, figsize=(15, 12))

# Price with Bollinger Bands
axes[0].plot(plot_df.index, plot_df['close'], label='Close', linewidth=1.5, color='black')
axes[0].plot(plot_df.index, plot_df['bb_upper'], label='BB Upper', linestyle='--', color='red', alpha=0.7)
axes[0].plot(plot_df.index, plot_df['bb_lower'], label='BB Lower', linestyle='--', color='blue', alpha=0.7)
axes[0].fill_between(plot_df.index, plot_df['bb_lower'], plot_df['bb_upper'], alpha=0.1)
axes[0].set_title('Price with Bollinger Bands', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RSI
axes[1].plot(plot_df.index, plot_df['rsi'], color='purple', linewidth=1.5)
axes[1].axhline(y=70, color='red', linestyle='--', label='Overbought', alpha=0.7)
axes[1].axhline(y=30, color='green', linestyle='--', label='Oversold', alpha=0.7)
axes[1].set_title('RSI (Relative Strength Index)', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 100)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# MACD
axes[2].plot(plot_df.index, plot_df['macd'], label='MACD', color='blue', linewidth=1.5)
axes[2].plot(plot_df.index, plot_df['macd_signal'], label='Signal', color='red', linewidth=1.5)
axes[2].bar(plot_df.index, plot_df['macd_hist'], label='Histogram', alpha=0.3, color='gray')
axes[2].set_title('MACD', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# Volume with ATR
ax3_twin = axes[3].twinx()
axes[3].bar(plot_df.index, plot_df['volume'], alpha=0.5, color='blue', label='Volume')
ax3_twin.plot(plot_df.index, plot_df['atr'], color='red', linewidth=1.5, label='ATR')
axes[3].set_title('Volume and ATR (Volatility)', fontsize=12, fontweight='bold')
axes[3].set_xlabel('Time', fontsize=11)
axes[3].legend(loc='upper left')
ax3_twin.legend(loc='upper right')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 These indicators help the RL agent understand:")
print("  - Overbought/oversold conditions (RSI)")
print("  - Trend strength and reversals (MACD)")
print("  - Price volatility (Bollinger Bands, ATR)")
print("  - Volume patterns (unusual activity)")

## Part 2: Risk Management Strategies

### Critical for Real Trading!

RL agents can be aggressive. Let's add safety!

In [ ]:
class RiskManager:
    """
    Risk management wrapper for RL trading agents
    """
    def __init__(self, initial_capital=100000, max_position_size=1.0,
                 max_drawdown_pct=0.20, stop_loss_pct=0.05, take_profit_pct=0.10):
        """
        Args:
            initial_capital: Starting capital
            max_position_size: Maximum position as fraction of capital
            max_drawdown_pct: Maximum drawdown before stopping (20%)
            stop_loss_pct: Stop loss per trade (5%)
            take_profit_pct: Take profit per trade (10%)
        """
        self.initial_capital = initial_capital
        self.current_capital = initial_capital
        self.peak_capital = initial_capital
        
        self.max_position_size = max_position_size
        self.max_drawdown_pct = max_drawdown_pct
        self.stop_loss_pct = stop_loss_pct
        self.take_profit_pct = take_profit_pct
        
        self.current_position = 0
        self.entry_price = None
        self.trades = []
    
    def calculate_position_size(self, capital, volatility, max_risk_per_trade=0.02):
        """
        Calculate position size based on volatility (Kelly Criterion inspired)
        
        Args:
            capital: Current capital
            volatility: ATR or std dev
            max_risk_per_trade: Max 2% risk per trade
        """
        risk_amount = capital * max_risk_per_trade
        position_size = risk_amount / (volatility + 1e-8)
        
        # Cap at max position size
        max_position = capital * self.max_position_size
        return min(position_size, max_position)
    
    def check_stop_loss(self, current_price):
        """
        Check if stop loss is triggered
        """
        if self.entry_price is None or self.current_position == 0:
            return False
        
        if self.current_position > 0:  # Long position
            loss_pct = (self.entry_price - current_price) / self.entry_price
            return loss_pct >= self.stop_loss_pct
        else:  # Short position
            loss_pct = (current_price - self.entry_price) / self.entry_price
            return loss_pct >= self.stop_loss_pct
    
    def check_take_profit(self, current_price):
        """
        Check if take profit is triggered
        """
        if self.entry_price is None or self.current_position == 0:
            return False
        
        if self.current_position > 0:  # Long position
            profit_pct = (current_price - self.entry_price) / self.entry_price
            return profit_pct >= self.take_profit_pct
        else:  # Short position
            profit_pct = (self.entry_price - current_price) / self.entry_price
            return profit_pct >= self.take_profit_pct
    
    def check_max_drawdown(self):
        """
        Check if maximum drawdown is exceeded
        """
        self.peak_capital = max(self.peak_capital, self.current_capital)
        drawdown = (self.peak_capital - self.current_capital) / self.peak_capital
        return drawdown >= self.max_drawdown_pct
    
    def filter_action(self, rl_action, current_price, atr):
        """
        Filter RL agent's action through risk management rules
        
        Args:
            rl_action: 0=Sell, 1=Hold, 2=Buy
            current_price: Current market price
            atr: Current ATR (volatility)
        
        Returns:
            filtered_action: Risk-adjusted action
        """
        # Check stop loss
        if self.check_stop_loss(current_price):
            print(f"⚠️  STOP LOSS triggered at {current_price:.2f}")
            return 1  # Force close (Hold/Neutral)
        
        # Check take profit
        if self.check_take_profit(current_price):
            print(f"✅ TAKE PROFIT triggered at {current_price:.2f}")
            return 1  # Force close
        
        # Check max drawdown
        if self.check_max_drawdown():
            print(f"🛑 MAX DRAWDOWN exceeded! Stopping trading.")
            return 1  # Force hold
        
        return rl_action


# Example usage
risk_manager = RiskManager(
    initial_capital=100000,
    max_position_size=0.5,  # Max 50% of capital per position
    max_drawdown_pct=0.20,  # Stop if 20% drawdown
    stop_loss_pct=0.05,     # 5% stop loss
    take_profit_pct=0.10    # 10% take profit
)

print("✅ Risk Manager initialized!")
print(f"\n🛡️  Risk Parameters:")
print(f"  - Max position: {risk_manager.max_position_size * 100}% of capital")
print(f"  - Stop loss: {risk_manager.stop_loss_pct * 100}%")
print(f"  - Take profit: {risk_manager.take_profit_pct * 100}%")
print(f"  - Max drawdown: {risk_manager.max_drawdown_pct * 100}%")

## Part 3: Custom Reward Functions

### Shape Agent Behavior with Better Rewards

In [ ]:
def calculate_sharpe_reward(returns, window=100):
    """
    Reward based on Sharpe ratio (risk-adjusted returns)
    """
    if len(returns) < window:
        return 0
    
    recent_returns = returns[-window:]
    mean_return = np.mean(recent_returns)
    std_return = np.std(recent_returns) + 1e-8
    sharpe = mean_return / std_return
    return sharpe

def calculate_sortino_reward(returns, window=100, target_return=0):
    """
    Reward based on Sortino ratio (penalizes downside volatility more)
    """
    if len(returns) < window:
        return 0
    
    recent_returns = returns[-window:]
    mean_return = np.mean(recent_returns)
    downside_returns = recent_returns[recent_returns < target_return]
    
    if len(downside_returns) == 0:
        return mean_return
    
    downside_std = np.std(downside_returns) + 1e-8
    sortino = (mean_return - target_return) / downside_std
    return sortino

def calculate_risk_adjusted_reward(profit, volatility, risk_aversion=0.5):
    """
    Combined reward: profit minus risk penalty
    
    Args:
        profit: Actual profit/loss
        volatility: Recent volatility (ATR or std)
        risk_aversion: How much to penalize risk (0=none, 1=high)
    """
    risk_penalty = risk_aversion * volatility
    return profit - risk_penalty

def calculate_profit_factor_reward(winning_trades, losing_trades):
    """
    Reward based on profit factor (gross profit / gross loss)
    """
    gross_profit = sum([t for t in winning_trades if t > 0])
    gross_loss = abs(sum([t for t in losing_trades if t < 0]))
    
    if gross_loss == 0:
        return gross_profit
    
    profit_factor = gross_profit / gross_loss
    return profit_factor - 1  # Center around 0

# Example comparison
print("💡 Reward Function Comparison:\n")
print("="*60)
print(f"{'Reward Type':<25} {'Focus':<35}")
print("="*60)
print(f"{'Simple Profit':<25} Basic PnL only")
print(f"{'Sharpe':<25} Risk-adjusted returns")
print(f"{'Sortino':<25} Penalizes downside more")
print(f"{'Risk-Adjusted':<25} Profit minus volatility penalty")
print(f"{'Profit Factor':<25} Ratio of wins to losses")
print("="*60)
print("\n🎯 Recommendation: Use **Sharpe** or **Sortino** for stable learning!")

## Part 4: Hyperparameter Tuning Guide

### Finding the Best Configuration

In [ ]:
# Hyperparameter tuning guidelines
tuning_guide = {
    'DQN': {
        'learning_rate': [0.0001, 0.0003, 0.001, 0.003],
        'batch_size': [32, 64, 128, 256],
        'gamma': [0.9, 0.95, 0.99],
        'epsilon_decay': [0.99, 0.995, 0.999],
        'network_size': [(64, 32), (128, 64), (256, 128)],
    },
    'PPO': {
        'learning_rate': [0.0001, 0.0003, 0.001],
        'batch_size': [64, 128, 256],
        'gamma': [0.95, 0.99, 0.999],
        'ratio_clip': [0.1, 0.2, 0.3],
        'lambda_entropy': [0.001, 0.01, 0.1],
        'horizon_len': [128, 256, 512],
    },
    'SAC': {
        'learning_rate': [0.0001, 0.0003, 0.001],
        'batch_size': [128, 256, 512],
        'gamma': [0.99, 0.995, 0.999],
        'tau': [0.001, 0.005, 0.01],
        'alpha': [0.1, 0.2, 0.5],
    }
}

print("🎯 Hyperparameter Tuning Guide\n")
print("="*70)

for algo, params in tuning_guide.items():
    print(f"\n{algo}:")
    for param, values in params.items():
        print(f"  {param:<20}: {values}")

print("\n" + "="*70)
print("\n💡 Tuning Tips:")
print("  1. Start with default values")
print("  2. Tune learning rate first (most important!)")
print("  3. Then batch size and network architecture")
print("  4. Finally, algorithm-specific parameters")
print("  5. Use grid search or Optuna for automation")
print("  6. Always validate on unseen data!")

## Part 5: Multi-Contract Trading

### Diversification for Better Returns

In [ ]:
# Compare all available contracts
contracts = ['MCL-1m', 'MGC-1m', 'mes-1m', 'ng', 'si']
contract_names = {
    'MCL-1m': 'Crude Oil',
    'MGC-1m': 'Gold',
    'mes-1m': 'S&P 500',
    'ng': 'Natural Gas',
    'si': 'Silver'
}

print("📊 Multi-Contract Strategy Guide\n")
print("="*70)

# Load and compare basic statistics
for contract in contracts:
    data_path = Path('./data') / contract / 'train.csv'
    if data_path.exists():
        df = pd.read_csv(data_path, index_col=0)
        df['returns'] = df['close'].pct_change()
        
        print(f"\n{contract} ({contract_names[contract]}):")
        print(f"  Mean Return:    {df['returns'].mean():.6f}")
        print(f"  Volatility:     {df['returns'].std():.6f}")
        print(f"  Sharpe (daily): {df['returns'].mean() / (df['returns'].std() + 1e-8):.3f}")

print("\n" + "="*70)
print("\n💡 Multi-Contract Benefits:")
print("  ✅ Diversification reduces risk")
print("  ✅ Different contracts have different patterns")
print("  ✅ Can train specialized agents per contract")
print("  ✅ Or train one agent on multiple contracts")
print("\n🎯 Strategy: Train separate agents, combine with portfolio optimization!")

## Part 6: Production Deployment Checklist

### From Notebook to Real Trading

In [ ]:
print("🚀 Production Deployment Checklist\n")
print("="*70)

checklist = [
    ("Data Pipeline", [
        "Real-time data feed integration",
        "Data quality checks",
        "Missing data handling",
        "Feature calculation latency < 100ms"
    ]),
    ("Model", [
        "Trained on multiple market conditions",
        "Walk-forward validation passed",
        "Stress tested on crashes/spikes",
        "Model versioning implemented"
    ]),
    ("Risk Management", [
        "Position size limits enforced",
        "Stop-loss mechanisms active",
        "Max drawdown circuit breaker",
        "Daily loss limit"
    ]),
    ("Infrastructure", [
        "Low-latency execution (<50ms)",
        "Redundant systems (failover)",
        "Monitoring and alerting",
        "Logging all trades and decisions"
    ]),
    ("Testing", [
        "Paper trading for 1+ month",
        "Performance matches backtest",
        "Slippage and fees included",
        "Stress scenarios tested"
    ]),
    ("Compliance", [
        "Regulatory requirements met",
        "Trade reporting automated",
        "Audit trail maintained",
        "Risk disclosures prepared"
    ])
]

for category, items in checklist:
    print(f"\n{category}:")
    for item in items:
        print(f"  ☐ {item}")

print("\n" + "="*70)
print("\n⚠️  CRITICAL REMINDERS:")
print("  1. NEVER go live without extensive paper trading")
print("  2. Start with SMALL position sizes")
print("  3. Monitor 24/7 for the first month")
print("  4. Have a kill switch ready")
print("  5. Backtest results ≠ live results (expect 20-30% degradation)")

## Summary: Becoming an Expert

### 🎉 Congratulations on Completing the Series!

You've journeyed from RL beginner to advanced practitioner!

### 📚 What You've Mastered:

#### Notebook 1 - Data Exploration:
- ✅ Understanding OHLCV data
- ✅ Technical indicators
- ✅ Data quality

#### Notebook 2 - DQN:
- ✅ RL fundamentals
- ✅ Value-based learning
- ✅ Q-networks

#### Notebook 3 - PPO:
- ✅ Policy gradients
- ✅ Actor-Critic
- ✅ Stable training

#### Notebook 4 - SAC:
- ✅ Maximum entropy RL
- ✅ Auto-tuning
- ✅ State-of-the-art performance

#### Notebook 5 - Advanced:
- ✅ Custom indicators
- ✅ Risk management
- ✅ Reward engineering
- ✅ Production deployment

### 🎯 Your RL Trading Toolkit:

1. **Algorithms**: DQN, PPO, SAC
2. **Indicators**: RSI, MACD, Bollinger, ATR, and more
3. **Risk Tools**: Position sizing, stop-loss, drawdown limits
4. **Analysis**: Sharpe, Sortino, profit factor
5. **Deployment**: Production checklist

### 🚀 Next Steps:

1. **Practice**: Try different contracts and algorithms
2. **Experiment**: Tune hyperparameters, test indicators
3. **Learn More**: Read papers on RL for finance
4. **Paper Trade**: Test strategies in real-time (no money)
5. **Join Community**: Share experiences, learn from others

### 📖 Recommended Resources:

- **TradeMaster Paper**: Latest research
- **Spinning Up in RL**: OpenAI's RL guide
- **Advances in Financial ML**: Marcos López de Prado
- **r/algotrading**: Reddit community
- **QuantConnect**: Backtesting platform

### 💡 Final Wisdom:

> "The market is a harsh teacher. Start small, learn constantly, and never risk more than you can afford to lose."

### 🏆 You're Now Ready:

- ✅ To build RL trading systems
- ✅ To understand academic papers
- ✅ To experiment with advanced techniques
- ✅ To contribute to the community

**Good luck with your trading journey! 🎯🚀**

---
### 🙏 Thank You!

Thank you for completing the "RL for Dummies" notebook series!

**Remember**: 
- Practice makes perfect
- Risk management is NOT optional
- Paper trade before live trading
- Keep learning and adapting

**Stay safe, trade smart! 💪**